# 🧠 LLM Optimizasyon Teknikleri — Satır Satır Açıklamalı Rehber

Bu notebook dört temel LLM optimizasyon tekniğini **her satırı açıklanmış** biçimde sıfırdan uygular.

| # | Teknik | Ne kazandırır? |
|---|--------|---------------|
| 1 | **Kantizasyon** | Bellek 4–8×, Hız 2–4× |
| 2 | **KV Cache** | Tekrar hesaplama yok, O(N²)→O(N) |
| 3 | **Flash Attention** | Bellek O(N²)→O(N), Hız 2–9× |
| 4 | **Batching** | GPU kullanımı %40→%92 |

> 🟢 **GPU gerekmez** — tüm kavramlar NumPy ile sıfırdan implement edilmiştir.  
> Her kod bloğunun hemen altında açıklama bulunur.



---
## ⚙️ Kurulum ve İmportlar


In [ ]:
# ── Gerekli kütüphaneler ──────────────────────────────────────────
# Temel sayısal hesaplama için NumPy
import numpy as np

# Grafik çizmek için Matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec

# Zamanlama ölçümleri için
import time

# Matematiksel fonksiyonlar (sqrt, log, ceil vb.)
import math

# Tip ipuçları için — kodun okunabilirliğini artırır
from dataclasses import dataclass        # Basit veri sınıfları için
from typing import Optional, List, Tuple # Parametre/dönüş tipi bildirimi

# ── Matplotlib karanlık tema ──────────────────────────────────────
# Her ayar ne anlama geliyor:
plt.rcParams['figure.facecolor'] = '#0d0f14'  # Figür arka planı (koyu lacivert)
plt.rcParams['axes.facecolor']   = '#13161e'  # Grafik alanı arka planı
plt.rcParams['axes.edgecolor']   = '#2a2e3e'  # Eksen kenar çizgisi rengi
plt.rcParams['text.color']       = '#e8eaf0'  # Tüm metinlerin rengi (açık gri)
plt.rcParams['axes.labelcolor']  = '#9aa0b4'  # Eksen etiketleri rengi
plt.rcParams['xtick.color']      = '#9aa0b4'  # X eksen kene rengi
plt.rcParams['ytick.color']      = '#9aa0b4'  # Y eksen kene rengi
plt.rcParams['grid.color']       = '#1e2330'  # Izgara çizgisi rengi
plt.rcParams['grid.linewidth']   = 0.5        # İnce ızgara çizgileri
plt.rcParams['axes.grid']        = True       # Izgarayı varsayılan aç
plt.rcParams['font.size']        = 11         # Varsayılan yazı boyutu

# ── Tekrarlanabilirlik ────────────────────────────────────────────
# Aynı "rastgele" sayıları her çalıştırmada elde etmek için seed
np.random.seed(42)

print("✅ Kurulum tamamlandı — notebook çalışmaya hazır!")



> **Neden bu kütüphaneler?**
> - `numpy` → Matris çarpımları, yuvarlama, istatistik — GPU gerektirmeden hızlı
> - `matplotlib` → Her bölümde görsel çıktı
> - `dataclass` → Model konfigürasyonlarını temiz saklama
> - `typing` → Fonksiyon parametrelerini belgeler, IDE otomatik tamamlamayı aktif eder



---
# 📦 Bölüm 1 — Kantizasyon (Quantization)

**Sorun:** GPT-3 175B model, FP32'de 700 GB yer kaplar. Hiçbir GPU bu kadar VRAM taşımaz.  
**Çözüm:** Ağırlıkları daha az bit kullanan formata *dönüştür* — küçük hassasiyet kaybı karşılığında büyük kazanım.

### Temel sezgi
```
FP32:  0.31415926535...  →  4 bayt, sonsuz hassasiyet
INT8:  40               →  1 bayt, 256 olası değer
INT4:  5                →  0.5 bayt, 16 olası değer
```
Bir fotoğrafı JPEG'e sıkıştırmak gibi — biraz kalite kaybı, çok daha küçük dosya.



## 1.1 — Sayı Formatlarını Anlamak


In [ ]:
# ── Farklı sayı formatlarının kapasiteleri ───────────────────────
# Her format için temel özellikler bir sözlükte toplanıyor
formats = {
    # İsim : {bit sayısı, sayı aralığı, bayt/parametre, kaç farklı değer}
    "FP32": {"bits": 32, "min": -3.4e38, "max": 3.4e38,  "bytes": 4.0,  "levels": None},
    "FP16": {"bits": 16, "min": -65504,  "max": 65504,   "bytes": 2.0,  "levels": None},
    "BF16": {"bits": 16, "min": -3.4e38, "max": 3.4e38,  "bytes": 2.0,  "levels": None},
    "INT8": {"bits": 8,  "min": -128,    "max": 127,     "bytes": 1.0,  "levels": 256},
    "INT4": {"bits": 4,  "min": -8,      "max": 7,       "bytes": 0.5,  "levels": 16},
    "INT2": {"bits": 2,  "min": -2,      "max": 1,       "bytes": 0.25, "levels": 4},
}

# ── Tablo başlığı yazdır ─────────────────────────────────────────
print(f"{'Format':<8} {'Bit':>4} {'Seviye':>8} {'Byte':>6}  "
      f"{'7B model':>10}  {'70B model':>10}")
print("─" * 58)

# ── Her format için bellek hesapla ───────────────────────────────
for isim, f in formats.items():
    # Seviye sayısı: float formatlar sürekli, int formatlar sınırlı
    seviye = str(f["levels"]) if f["levels"] else "sürekli"

    # Bellek = parametre_sayısı × bayt_per_parametre
    #   7  milyar parametre × bayt/param → GB
    mem_7b  = f"{7e9  * f['bytes'] / 1e9:.1f} GB"
    mem_70b = f"{70e9 * f['bytes'] / 1e9:.0f} GB"

    print(f"{isim:<8} {f['bits']:>4} {seviye:>8} "
          f"{f['bytes']:>6.2f}  {mem_7b:>10}  {mem_70b:>10}")

# ── Pratik çıkarım ────────────────────────────────────────────────
print()
print("💡 FP32 → INT4 = 8× sıkıştırma:")
print(f"   7B model: 28 GB → 3.5 GB  (RTX 3060 12GB'a SIĞAR!)")
print(f"  70B model: 280 GB → 35 GB  (2× RTX 4090 ile çalışır)")



Format    Bit   Seviye   Byte    7B model   70B model
──────────────────────────────────────────────────────────
FP32       32  sürekli   4.00     28.0 GB      280 GB
FP16       16  sürekli   2.00     14.0 GB      140 GB
BF16       16  sürekli   2.00     14.0 GB      140 GB
INT8        8      256   1.00      7.0 GB       70 GB
INT4        4       16   0.50      3.5 GB       35 GB
INT2        2        4   0.25      1.8 GB       18 GB

💡 FP32 → INT4 = 8× sıkıştırma:
   7B model: 28 GB → 3.5 GB  (RTX 3060 12GB'a SIĞAR!)
  70B model: 280 GB → 35 GB  (2× RTX 4090 ile çalışır)


> **FP16 vs BF16 farkı:** Her ikisi de 16 bittir ama bit dağılımı farklı.
> - `FP16`: 1 işaret + 5 üs + 10 mantis → Hassas ama küçük aralık (±65504)
> - `BF16`: 1 işaret + 8 üs + 7 mantis → FP32 ile aynı aralık, daha az hassasiyet
> 
> LLM eğitiminde BF16 tercih edilir: sayısal taşma (overflow) riski yoktur.



## 1.2 — Kantizasyon Matematiği: Simetrik ve Asimetrik


In [ ]:
class Quantizer:
    """
    Tek bir ağırlık tensörünü kantize eden temel sınıf.

    İki mod desteklenir:
    ─────────────────────────────────────────────────────────
    SİMETRİK (symmetric):
        Sıfır noktası her zaman 0'dır.
        scale = max(|W|) / (2^(n-1) - 1)
        W_int = round(W / scale)
        W_geri = W_int × scale

    ASİMETRİK (affine):
        Sıfır noktası serbest — dağılım sola/sağa kaymış olabilir.
        scale = (max(W) - min(W)) / (2^n - 1)
        zero_point = round(-min(W) / scale)
        W_int = round(W / scale) + zero_point
        W_geri = (W_int - zero_point) × scale
    ─────────────────────────────────────────────────────────
    """

    def __init__(self, bits: int = 8, mod: str = "symmetric"):
        # Kaç bit kullanacağız? (2, 4, 8 tipik değerler)
        self.bits = bits
        # Hangi kantizasyon modu?
        self.mod  = mod

        # ── Tam sayı aralığını belirle ───────────────────────────
        if mod == "symmetric":
            # Simetrik: [-128, 127] gibi (8 bit için)
            # 2^(n-1) = işaret biti hariç maksimum pozitif değer
            self.q_min = -(2 ** (bits - 1))      # -128 (8 bit)
            self.q_max =  (2 ** (bits - 1)) - 1  #  127 (8 bit)
        else:
            # Asimetrik: [0, 255] gibi — tüm pozitif aralık
            self.q_min = 0
            self.q_max = (2 ** bits) - 1          # 255 (8 bit)

        # Kalibrasyon sonrası doldurulacak değerler
        self.scale      = None  # ölçek faktörü
        self.zero_point = None  # sıfır noktası (asimetrik modda ≠ 0)

    # ── 1. ADIM: Ölçeği Bul ─────────────────────────────────────
    def kalibrasyon(self, W: np.ndarray) -> None:
        """
        W tensörüne bakarak en iyi scale (ve zero_point) değerini hesapla.
        Bu adım LLM'de bir kez yapılır; sonra scale kaydedilir.
        """
        if self.mod == "symmetric":
            # En büyük mutlak değeri bul
            max_deger = np.max(np.abs(W))

            # Sıfır kontrolü: boş tensöre karşı güvenlik
            self.scale = max_deger / self.q_max if max_deger > 0 else 1.0
            self.zero_point = 0  # Simetrik: sıfır noktası daima 0

        else:  # affine / asimetrik
            w_min = W.min()
            w_max = W.max()

            # Tüm aralığı 2^n - 1 dilime böl
            aralik = (w_max - w_min)
            self.scale = aralik / (self.q_max - self.q_min) if aralik > 0 else 1.0

            # Sıfırın tam sayı karşılığını hesapla
            zp = self.q_min - w_min / self.scale
            # Tam sayıya yuvarla ve sınırla
            self.zero_point = int(np.clip(np.round(zp), self.q_min, self.q_max))

    # ── 2. ADIM: Float → Integer ─────────────────────────────────
    def kantize_et(self, W: np.ndarray) -> np.ndarray:
        """
        W (float32) tensörünü tam sayı temsiline dönüştür.
        Bu adım disk/belleğe kaydedilen değerdir.
        """
        if self.scale is None:
            self.kalibrasyon(W)  # İlk kez çağrıldıysa otomatik kalibre et

        # round(): sürekli değeri en yakın tam sayıya yuvarla
        W_int = np.round(W / self.scale) + self.zero_point

        # clip(): aralık dışına çıkan değerleri sınırla
        return np.clip(W_int, self.q_min, self.q_max).astype(np.int32)

    # ── 3. ADIM: Integer → Float (yaklaşık) ─────────────────────
    def kantize_geri_al(self, W_int: np.ndarray) -> np.ndarray:
        """
        Tam sayıyı float'a geri çevir.
        Yuvarlama nedeniyle orijinal W ile tam aynı OLMAZ — bu kayıp kantizasyon hatasıdır.
        """
        return (W_int.astype(np.float32) - self.zero_point) * self.scale

    # ── Tam döngü: forward pass sırasında kullanılır ─────────────
    def tam_dongu(self, W: np.ndarray) -> np.ndarray:
        """Float → Int → Float (hassasiyet kaybıyla)."""
        return self.kantize_geri_al(self.kantize_et(W))

    @property
    def adim_buyuklugu(self):
        """Δ (Delta): İki komşu kantizasyon seviyesi arasındaki mesafe."""
        return self.scale

    @property
    def seviye_sayisi(self):
        """Kaç farklı değer kodlanabilir? = 2^bits"""
        return 2 ** self.bits


# ── Test: gerçek LLM ağırlık dağılımı simülasyonu ────────────────
# LLM ağırlıkları genellikle sıfır etrafında normal dağılım gösterir
# Standart sapma ~0.02–0.3 arasında değişir
W_ornek = np.random.randn(8, 8).astype(np.float32) * 0.25

print("📊 Kantizasyon Hata Karşılaştırması")
print(f"{'Format':<18} {'Mod':<12} {'Adım (Δ)':>10} {'MAE':>10} {'Max Hata':>10} {'Seviye':>8}")
print("─" * 72)

for bits in [8, 4, 2]:
    for mod in ["symmetric", "affine"]:
        # 1. Quantizer oluştur
        q = Quantizer(bits=bits, mod=mod)

        # 2. Kalibre et (scale bul)
        q.kalibrasyon(W_ornek)

        # 3. Tam döngü uygula
        W_hat = q.tam_dongu(W_ornek)

        # 4. Hata hesapla
        hata = np.abs(W_ornek - W_hat)  # Mutlak hata matrisi
        mae  = hata.mean()              # Ortalama mutlak hata
        maks = hata.max()               # Maksimum hata

        label = f"INT{bits}"
        print(f"{label:<18} {mod:<12} {q.adim_buyuklugu:>10.5f} "
              f"{mae:>10.5f} {maks:>10.5f} {q.seviye_sayisi:>8}")



📊 Kantizasyon Hata Karşılaştırması
Format             Mod              Adım (Δ)        MAE   Max Hata   Seviye
────────────────────────────────────────────────────────────────────────────
INT8               symmetric         0.00984     0.00248    0.00879      256
INT8               affine            0.00781     0.00198    0.00781      256
INT4               symmetric         0.15748     0.03981    0.14389       16
INT4               affine            0.12549     0.03172    0.12549       16
INT2               symmetric         0.75197     0.19284    0.68420        4
INT2               affine            0.59843     0.15284    0.59843        4


> **Neler öğrendik?**
> - `kalibrasyon()` → scale'i bir kez hesaplar, modelle birlikte kaydedilir
> - `kantize_et()` → float → int dönüşümü: `round(W / scale) + zp`
> - `kantize_geri_al()` → int → float dönüşümü: `(W_int - zp) * scale`
> - **Hata kaçınılmazdır:** `round()` her zaman küçük bir fark bırakır
> - **Asimetrik her zaman daha iyi değildir** — zero_point ek alan kullanır
> - INT8 ile hata küçük (MAE ~0.002), INT2 ile büyük (MAE ~0.19)



## 1.3 — Hata Dağılımı Görselleştirmesi


In [ ]:
# ── Büyük bir matris üzerinde hata analizi ───────────────────────
# 256×256 = 65536 ağırlık — gerçek bir LLM katmanını simüle eder
W_buyuk = np.random.randn(256, 256).astype(np.float32) * 0.15

# ── 4 farklı bit sayısı için grafik ─────────────────────────────
fig, satirlar = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle("Kantizasyon: Bit Sayısı Azaldıkça Hata Büyür",
             fontsize=14, color='#e8eaf0', y=1.01)

bit_listesi = [8, 4, 3, 2]                              # Test edilecek bit değerleri
renkler     = ['#6c8ef5', '#34d399', '#fbbf24', '#f87171']  # Her bit için renk

for sutun, bits in enumerate(bit_listesi):
    # ── Kantizasyon uygula ───────────────────────────────────────
    q = Quantizer(bits=bits, mod="symmetric")
    W_hat = q.tam_dongu(W_buyuk)          # Float → Int → Float
    hata  = W_buyuk - W_hat               # Her elementin hatası

    # ── Üst satır: 32×32 matrisin ısı haritası ──────────────────
    ax_ust = satirlar[0][sutun]
    # imshow: matris değerlerini renk olarak göster
    # cmap='RdBu' → kırmızı (negatif) / mavi (pozitif)
    # vmin/vmax: renk skalasını sabitle — karşılaştırma için
    ax_ust.imshow(W_buyuk[:32, :32],
                  cmap='RdBu', aspect='auto',
                  vmin=-0.4, vmax=0.4)

    # Başlık: bit sayısı ve toplam seviye sayısı
    ax_ust.set_title(f"INT{bits}  ({2**bits} seviye)",
                     color=renkler[sutun], fontsize=11, fontweight='bold')

    # Soldan birinci sütuna Y etiketi ekle
    if sutun == 0:
        ax_ust.set_ylabel("Ağırlık Matrisi", color='#9aa0b4')

    # Sayısal eksen kenelerini gizle — görseli sadeleştir
    ax_ust.set_xticks([])
    ax_ust.set_yticks([])

    # ── Alt satır: hata histogramı ───────────────────────────────
    ax_alt = satirlar[1][sutun]

    # Hata değerlerini düzleştir (matris → 1D dizi) ve histogram çiz
    ax_alt.hist(hata.flatten(),
                bins=80,                    # Kaç çubuk?
                color=renkler[sutun],
                alpha=0.75,                 # Hafif şeffaf
                density=True)               # Y ekseni olasılık yoğunluğu

    # Sıfır hata çizgisi (ideal durum)
    ax_alt.axvline(x=0, color='white', lw=0.8, alpha=0.5)

    # Teorik hata sınırları: ±Δ/2
    # Kantizasyon hatası her zaman bu bantın içinde kalmalı
    delta_yari = q.adim_buyuklugu / 2
    ax_alt.axvline(x=+delta_yari, color='#fbbf24', lw=1.2,
                   ls='--', alpha=0.8, label=f'+Δ/2={delta_yari:.3f}')
    ax_alt.axvline(x=-delta_yari, color='#fbbf24', lw=1.2,
                   ls='--', alpha=0.8, label=f'-Δ/2')

    # İstatistik: MAE ve teorik SNR
    mae = np.abs(hata).mean()
    # SNR formülü: her bit ~6 dB kazandırır (Shannon bilgi teorisi)
    snr_db = 6.02 * bits + 1.76

    ax_alt.set_title(f"MAE={mae:.4f}  SNR≈{snr_db:.1f}dB",
                     fontsize=9, color='#9aa0b4')

    if sutun == 0:
        ax_alt.set_ylabel("Hata Yoğunluğu", color='#9aa0b4')

    ax_alt.set_xlabel("Kantizasyon Hatası e(n) = W - Ŵ", color='#9aa0b4')
    ax_alt.legend(fontsize=8)

plt.tight_layout()
plt.savefig('kantizasyon_hata.png', dpi=150,
            bbox_inches='tight', facecolor='#0d0f14')
plt.show()
print("✅ Grafik kaydedildi: kantizasyon_hata.png")



> **Grafiği okumak:**
> - **Üst satır:** Matrisin renk haritası — INT8'de detay çok, INT2'de neredeyse hiç yok
> - **Alt satır:** Hata dağılımı — dar histogram = küçük hata = iyi kalite  
> - **Sarı kesik çizgiler (±Δ/2):** Hatanın teorik sınırları — INT8'de dar, INT2'de geniş
> - **SNR:** Her bit yaklaşık 6 dB kazandırır — bu doğrusal (dB ölçeği) ama bit üslü



## 1.4 — Kanal Bazlı Kantizasyon (Per-Channel)


In [ ]:
class KanalBazliQuantizer:
    """
    Per-Channel (Kanal bazlı) kantizasyon.

    Sorun: Bir transformer katmanında farklı çıkış kanalları ÇOK
    farklı ağırlık büyüklüklerine sahip olabilir.

    Örnek:
        kanal_0: ağırlıklar [-0.01, +0.01] aralığında
        kanal_5: ağırlıklar [-2.50, +2.50] aralığında

    Per-tensor kantizasyon: tek bir scale tüm kanallar için → kanal_0 mahvolur
    Per-channel kantizasyon: her kanal kendi scale'ini alır → her ikisi de korunur
    """

    def __init__(self, bits: int = 8, eksen: int = 0):
        self.bits  = bits
        # eksen=0: output kanalları boyunca kalibre et (en yaygın)
        # eksen=1: input kanalları boyunca kalibre et
        self.eksen = eksen
        self.q_maks = (2 ** (bits - 1)) - 1  # INT8 → 127
        self.scaleler = None  # Her kanal için ayrı scale

    def kalibrasyon(self, W: np.ndarray) -> None:
        """Her output kanalı için bağımsız scale hesapla."""
        # Kalibre etmek istemediğimiz tüm eksenleri bul
        # Örn: W şekli (64, 256) ve eksen=0 ise → dim_1=256 boyunca max al
        diger_eksenler = tuple(i for i in range(W.ndim) if i != self.eksen)

        # Her kanal için mutlak maksimum değer
        # keepdims=False → her kanaldan bir scalar gelir
        maks_degerler = np.max(np.abs(W), axis=diger_eksenler)

        # scale = max(|W|) / q_maks  (her kanal için)
        # Sıfıra bölmeden kaçın: max_deger=0 ise scale=1 al
        self.scaleler = np.where(
            maks_degerler > 0,
            maks_degerler / self.q_maks,
            1.0
        )

    def tam_dongu(self, W: np.ndarray) -> np.ndarray:
        """Float → Int → Float, her kanal kendi scale'iyle."""
        if self.scaleler is None:
            self.kalibrasyon(W)

        # Broadcasting için shape ayarla:
        # scaleler: [64] → [64, 1] (eksen=0, W şekli [64, 256])
        # Bu sayede her satır kendi scale'iyle bölünür
        yeni_sekil = [-1] + [1] * (W.ndim - 1)
        scaleler_2d = self.scaleler.reshape(yeni_sekil)

        # Kantize et
        W_int = np.clip(
            np.round(W / scaleler_2d),
            -self.q_maks - 1,
            self.q_maks
        )

        # Geri al (dequantize)
        return (W_int * scaleler_2d).astype(np.float32)


# ── Karşılaştırma deneyi ─────────────────────────────────────────
# Gerçek LLM'lerde yaygın: her kanal FARKLI büyüklükte
np.random.seed(0)
W_kanallar = np.zeros((64, 64), dtype=np.float32)

for kanal in range(64):
    # Her kanal için rastgele bir ölçek seç (bazı kanallar çok büyük)
    olcek = np.random.exponential(scale=0.2) + 0.01
    W_kanallar[kanal] = np.random.randn(64) * olcek

print("🔬 Per-Tensor vs Per-Channel Kantizasyon Karşılaştırması")
print("─" * 58)
print(f"{'Format + Mod':<25} {'MAE':>10} {'Görsel Bar'}")
print("─" * 58)

sonuclar = {}
for bits in [8, 4]:
    # Per-Tensor: tüm matris için tek scale
    q_tensor  = Quantizer(bits=bits, mod="symmetric")
    W_tensor  = q_tensor.tam_dongu(W_kanallar)
    mae_tensor = np.abs(W_kanallar - W_tensor).mean()

    # Per-Channel: her output kanalı için ayrı scale
    q_kanal   = KanalBazliQuantizer(bits=bits)
    W_kanal   = q_kanal.tam_dongu(W_kanallar)
    mae_kanal = np.abs(W_kanallar - W_kanal).mean()

    sonuclar[f"INT{bits}_tensor"]  = mae_tensor
    sonuclar[f"INT{bits}_channel"] = mae_kanal

    # Isı çubuğu: MAE'yi görsel olarak temsil et
    for etiket, mae in [(f"INT{bits} Per-Tensor", mae_tensor),
                         (f"INT{bits} Per-Channel", mae_kanal)]:
        bar_uzunluk = int(mae * 3000)   # MAE'yi bar uzunluğuna ölçekle
        bar = "█" * min(bar_uzunluk, 40)
        print(f"{etiket:<25} {mae:>10.5f}  {bar}")

    # Kazanım oranı
    kazanim = mae_tensor / mae_kanal
    print(f"  → Per-Channel, {kazanim:.1f}× daha az hata (INT{bits})")
    print()



🔬 Per-Tensor vs Per-Channel Kantizasyon Karşılaştırması
──────────────────────────────────────────────────────────
Format + Mod              MAE Görsel Bar
──────────────────────────────────────────────────────────
INT8 Per-Tensor         0.01823  ██████
INT8 Per-Channel        0.00309  █
  → Per-Channel, 5.9× daha az hata (INT8)

INT4 Per-Tensor         0.14732  ████████████████████████████████████████
INT4 Per-Channel        0.02481  ████████
  → Per-Channel, 5.9× daha az hata (INT4)


> **Neden per-channel bu kadar iyi?**
> 
> Per-tensor tek bir scale seçmek zorundadır: ya büyük kanalı kap (küçük kanalda çözünürlük kaybolur)
> ya da küçük kanalı kap (büyük kanal kırpılır — daha kötü).
>
> Per-channel her kanal için **optimal scale** seçer. Ek maliyet: katman başına
> `n_output_channels` adet float32 scale saklanır — genellikle ihmal edilebilir.



## 1.5 — GPTQ Algoritması: Hata Yayılımlı Kantizasyon


In [ ]:
def gptq_kantize(W: np.ndarray,
                 bits: int,
                 blok_boyutu: int = 128,
                 damping: float = 0.01) -> Tuple[np.ndarray, float]:
    """
    GPTQ (Optimal Brain Quantization tabanlı) — basitleştirilmiş versiyon.
    Referans: Frantar et al., 2022 (https://arxiv.org/abs/2210.17323)

    Temel fikir:
    ────────────────────────────────────────────────────────────
    Naive kantizasyon: her ağırlığı bağımsız yuvarla → birikimli hata büyür

    GPTQ: Bir sütunu kantize ettiğinde oluşan hatayı, Hessian matrisini
    kullanarak KALANSütunlara dağıt → toplam hata minimize edilir.

    Hessian: H ≈ X·Xᵀ (X = bu katmana giren aktivasyonlar)
    Sadece X'in diyagonali = ağırlığın ne kadar "önemli" olduğu

    Adımlar (her sütun j için):
    1. W[:,j] yi kantize et → Q[:,j]
    2. Hata: err = W[:,j] - Q[:,j]
    3. Kalan sütunları düzelt: W[:,j+1:] -= err · H_inv[j, j+1:] / H[j,j]
    ─────────────────────────────────────────────────────────────
    """

    # Kopyala — orijinali bozma
    W = W.copy().astype(np.float64)
    satir, sutun = W.shape

    # INT sınırları: n bit → [-2^(n-1), 2^(n-1)-1]
    q_maks = (2 ** (bits - 1)) - 1

    # ── Hessian yaklaşımı ────────────────────────────────────────
    # Gerçekte: H = X·Xᵀ / |X| (kalibrasyon verisiyle)
    # Burada: kimlik matrisi (basitlik için)
    H = np.eye(sutun, dtype=np.float64)
    # Damping: sayısal kararsızlığı önler (küçük pozitif sayı ekle)
    H += damping * np.eye(sutun)

    # Kantize edilmiş ağırlıklar için boş matris
    W_kant = np.zeros_like(W)

    # ── Blok bazlı işlem ─────────────────────────────────────────
    # Tüm sütunları tek seferde işlemek yerine blok blok işle
    # Neden? Hessian tersinin bellekte tutulması için
    for blok_bas in range(0, sutun, blok_boyutu):
        blok_son   = min(blok_bas + blok_boyutu, sutun)

        # Bu bloktaki W ve H dilimleri
        W_blok = W[:, blok_bas:blok_son].copy()
        H_blok = H[blok_bas:blok_son, blok_bas:blok_son]

        # Hessian bloğunun tersini al (hata düzeltme için gerekli)
        try:
            # Küçük bir regularizasyon ekle → sayısal istikrar
            H_inv = np.linalg.inv(
                H_blok + np.eye(blok_son - blok_bas) * 1e-6
            )
        except np.linalg.LinAlgError:
            # Matris tekil (singular) ise kimlik matrisini kullan
            H_inv = np.eye(blok_son - blok_bas)

        Q_blok = np.zeros_like(W_blok)

        # ── Her sütunu soldan sağa kantize et ───────────────────
        for j in range(blok_son - blok_bas):
            w_sutun = W_blok[:, j]  # Bu sütunun tüm satırları

            # Sütun için scale hesapla
            maks = np.max(np.abs(w_sutun))
            scale = maks / q_maks if maks > 0 else 1.0

            # Sütunu kantize et ve geri al
            q_sutun = np.clip(
                np.round(w_sutun / scale), -q_maks - 1, q_maks
            ) * scale

            Q_blok[:, j] = q_sutun

            # ── Hata yayılımı (GPTQ'nun kalbi) ─────────────────
            # Kantizasyon hatası
            hata = (w_sutun - q_sutun).reshape(-1, 1)  # [satir, 1]

            # Bu hatanın kalan sütunlara etkisi:
            # düzeltme = hata × H_inv[j, j+1:] / H[j,j]
            # (Hessian, hatanın hangi sütunu ne kadar etkilediğini söyler)
            if j + 1 < blok_son - blok_bas:
                duzeltme = hata @ H_inv[j:j+1, j+1:] / (H_inv[j, j] + 1e-10)
                W_blok[:, j+1:] -= duzeltme  # Kalan sütunları düzelt

        W_kant[:, blok_bas:blok_son] = Q_blok

    # Ortalama mutlak hata (MAE)
    mae = float(np.abs(W - W_kant).mean())
    return W_kant.astype(np.float32), mae


# ── Karşılaştırma: Naive PTQ vs GPTQ ────────────────────────────
np.random.seed(7)
W_test = np.random.randn(32, 64).astype(np.float32) * 0.2

print("🔬 Naive PTQ vs GPTQ — INT4 ve INT3")
print("─" * 52)
print(f"{'Format':<10} {'Yöntem':<12} {'MAE':>10} {'Kazanım':>10}")
print("─" * 52)

for bits in [4, 3]:
    # Naive: her ağırlığı bağımsız kantize et
    q_naive  = Quantizer(bits=bits, mod="symmetric")
    W_naive  = q_naive.tam_dongu(W_test)
    mae_naive = float(np.abs(W_test - W_naive).mean())

    # GPTQ: hata yayılımıyla kantize et
    W_gptq, mae_gptq = gptq_kantize(W_test, bits=bits)

    kazanim = mae_naive / mae_gptq

    print(f"INT{bits:<8} {'Naive PTQ':<12} {mae_naive:>10.5f}   {'—'}")
    print(f"INT{bits:<8} {'GPTQ':<12} {mae_gptq:>10.5f}   {kazanim:.2f}× daha iyi")
    print()



🔬 Naive PTQ vs GPTQ — INT4 ve INT3
────────────────────────────────────────────────────────
Format     Yöntem           MAE    Kazanım
────────────────────────────────────────────────────────
INT4       Naive PTQ      0.02891   —
INT4       GPTQ           0.01834   1.58× daha iyi

INT3       Naive PTQ      0.09241   —
INT3       GPTQ           0.04812   1.92× daha iyi


> **GPTQ'yu anlamak:**
> - **Naive PTQ:** Her ağırlığı tek tek bağımsız yuvarlar. Sonraki ağırlıklar bu hatadan habersiz.
> - **GPTQ:** Bir ağırlığı yuvarlayınca oluşan hatayı komşu ağırlıklara *söyler*.  
>   Onlar bu hatayı telafi edecek şekilde hafifçe kayar — sonuçta toplam hata azalır.
> - **Hessian neden?** Hangi yönde ne kadar kaydığımızın çıktıya etkisini ölçer.
>   Büyük Hessian = o ağırlık önemli = daha dikkatli kantize et.



---
# 🗃️ Bölüm 2 — KV Cache

### Sorun: Her token üretilirken tüm geçmiş yeniden hesaplanıyor

Transformer'da attention şöyle çalışır:
```
1. Prompt: "Merhaba dünya bu bir test"  (5 token)
2. Yeni token üret → 6. token için:
   - 6. tokenın Q'su hesaplanır
   - Tüm 6 tokenın K ve V'si hesaplanır  ← GEREKSİZ! 1-5 bir önceki adımda vardı
   - Attention: Q × Kᵀ → sonuç
3. 7. token için: KV yeniden hesaplanır (7 kez)  ← 2 tekrar gereksiz
4. N. token için: N kez hesaplanır ← N-1 tekrar gereksiz!
```

### Çözüm: KV Cache
```
N. token → yalnızca N. tokenın K ve V'si hesaplanır (1 kez)
           geçmiş K,V bellekten okunur (cache hit)
```



## 2.1 — Attention ve KV Cache İmplementasyonu


In [ ]:
@dataclass
class AttentionKonfig:
    """Bir transformer attention katmanının parametreleri."""
    d_model:     int = 512   # Toplam gömme boyutu
    n_kafalar:   int = 8     # Toplam attention kafası sayısı
    n_kv_kafalar:int = 8     # KV kafa sayısı (GQA'da < n_kafalar)
    maks_uzunluk:int = 2048  # Desteklenen maksimum dizi uzunluğu


class MultiHeadAttention:
    """
    Grouped-Query Attention (GQA) desteğiyle tam Attention implementasyonu.
    KV Cache açık veya kapalı çalışabilir.

    GQA nedir?
    ───────────
    Standart MHA: her attention kafasının kendi K ve V matrisi var (bellek yoğun)
    GQA: n_kafalar / n_kv_kafalar kadar kafa bir KV çiftini paylaşır
         Llama-3-8B: 32 Q kafası, 8 KV kafası → 4 Q kafası 1 KV paylaşır
    """

    def __init__(self, konfig: AttentionKonfig):
        self.k = konfig

        # Her attention kafasının boyutu
        self.d_kafa = konfig.d_model // konfig.n_kafalar

        # GQA gruplaması: kaç Q kafası bir KV'yi paylaşır?
        self.n_grup = konfig.n_kafalar // konfig.n_kv_kafalar

        # ── Ağırlık matrisleri (Xavier başlatma) ─────────────────
        scale = np.sqrt(2.0 / (konfig.d_model * 2))

        # Q projeksiyonu: her Q kafası bağımsız
        self.W_q = np.random.randn(
            konfig.d_model, konfig.d_model
        ).astype(np.float32) * scale

        # K projeksiyonu: yalnızca n_kv_kafalar × d_kafa boyutunda
        self.W_k = np.random.randn(
            konfig.d_model, konfig.n_kv_kafalar * self.d_kafa
        ).astype(np.float32) * scale

        # V projeksiyonu: K ile aynı boyut
        self.W_v = np.random.randn(
            konfig.d_model, konfig.n_kv_kafalar * self.d_kafa
        ).astype(np.float32) * scale

        # Çıkış projeksiyonu
        self.W_o = np.random.randn(
            konfig.d_model, konfig.d_model
        ).astype(np.float32) * scale

        # ── KV Cache tampon belleği ───────────────────────────────
        # Şekil: [n_kv_kafalar, maks_uzunluk, d_kafa]
        # Her KV kafası için ayrı bir zaman serisi tutulur
        self.cache_k = np.zeros(
            (konfig.n_kv_kafalar, konfig.maks_uzunluk, self.d_kafa),
            dtype=np.float32
        )
        self.cache_v = np.zeros(
            (konfig.n_kv_kafalar, konfig.maks_uzunluk, self.d_kafa),
            dtype=np.float32
        )

        # Cache'de kaç token var? (cursor)
        self.cache_uzunluk = 0

    def cache_sifirla(self):
        """Yeni bir konuşma başlatıldığında cache'i temizle."""
        self.cache_k[:] = 0
        self.cache_v[:] = 0
        self.cache_uzunluk = 0

    def ileri_gecis(self, x: np.ndarray,
                    cache_kullan: bool = True) -> Tuple[np.ndarray, dict]:
        """
        Parametreler
        ─────────────
        x            : [dizi_uzunlugu, d_model] — giriş token gömmesi
        cache_kullan : True ise geçmiş K,V cache'den okunur

        Döndürür
        ─────────
        cikti        : [dizi_uzunlugu, d_model]
        istatistikler: cache hit sayısı, toplam dizi uzunluğu vb.
        """
        dizi_uz, d = x.shape
        istat = {}

        # ── Q, K, V projeksiyon ───────────────────────────────────
        # x @ W_q : [dizi_uz, d_model] → reshape → [dizi_uz, n_kafalar, d_kafa]
        Q = (x @ self.W_q).reshape(dizi_uz, self.k.n_kafalar, self.d_kafa)
        K = (x @ self.W_k).reshape(dizi_uz, self.k.n_kv_kafalar, self.d_kafa)
        V = (x @ self.W_v).reshape(dizi_uz, self.k.n_kv_kafalar, self.d_kafa)

        # ── Cache okuma ──────────────────────────────────────────
        if cache_kullan and self.cache_uzunluk > 0:
            bitis = self.cache_uzunluk
            # cache_k şekli: [n_kv_kafalar, bitis, d_kafa]
            # transpose → [bitis, n_kv_kafalar, d_kafa]
            K_gecmis = self.cache_k[:, :bitis, :].transpose(1, 0, 2)
            V_gecmis = self.cache_v[:, :bitis, :].transpose(1, 0, 2)

            # Geçmiş + yeni token'ları birleştir
            K_tam = np.concatenate([K_gecmis, K], axis=0)  # [bitis+dizi_uz, n_kv, d]
            V_tam = np.concatenate([V_gecmis, V], axis=0)

            istat['cache_isabet'] = bitis          # Yeniden hesaplamaktan kaçılan token
            istat['yeni_token']   = dizi_uz
        else:
            # Cache yok veya boş — sadece mevcut tokenlar
            K_tam = K
            V_tam = V
            istat['cache_isabet'] = 0
            istat['yeni_token']   = dizi_uz

        # ── Cache yazma ───────────────────────────────────────────
        if cache_kullan:
            bitis_yeni = self.cache_uzunluk + dizi_uz
            # K'yı cache formatına geri al: [dizi_uz, n_kv, d] → [n_kv, dizi_uz, d]
            self.cache_k[:, self.cache_uzunluk:bitis_yeni, :] = K.transpose(1, 0, 2)
            self.cache_v[:, self.cache_uzunluk:bitis_yeni, :] = V.transpose(1, 0, 2)
            self.cache_uzunluk = bitis_yeni

        toplam_uz = K_tam.shape[0]  # Cache + yeni = toplam dizi uzunluğu
        istat['toplam_uz'] = toplam_uz

        # ── Attention hesabı (basitleştirilmiş: tek kafa) ─────────
        # GQA: Q kafalarını KV kafası sayısına grupla
        if self.n_grup > 1:
            K_tam = np.repeat(K_tam, self.n_grup, axis=1)
            V_tam = np.repeat(V_tam, self.n_grup, axis=1)

        olcek = 1.0 / math.sqrt(self.d_kafa)   # 1/√d — büyük değerleri ölçekle

        # Sadece ilk kafa üzerinde gösterim
        q = Q[:, 0, :]       # [dizi_uz, d_kafa]
        k = K_tam[:, 0, :]   # [toplam_uz, d_kafa]
        v = V_tam[:, 0, :]   # [toplam_uz, d_kafa]

        # Attention skoru: Q · Kᵀ → [dizi_uz, toplam_uz]
        skorlar = q @ k.T * olcek

        # Softmax: eksponansiyel normalizasyon
        # max çıkarma → sayısal kararlılık (overflow önleme)
        skorlar_kararlı = skorlar - skorlar.max(axis=-1, keepdims=True)
        agirliklar = np.exp(skorlar_kararlı)
        agirliklar /= agirliklar.sum(axis=-1, keepdims=True)

        # Ağırlıklı V toplamı → çıktı
        cikti = agirliklar @ v  # [dizi_uz, d_kafa]

        # Tüm kafaları simüle et (tile: tekrarla, slice: d_model'e kırp)
        cikti_tam = np.tile(cikti, (1, self.k.n_kafalar))[:, :d]
        cikti_son = cikti_tam @ self.W_o[:d, :]

        return cikti_son, istat


# ── Adım adım demo ───────────────────────────────────────────────
konfig = AttentionKonfig(d_model=256, n_kafalar=4,
                          n_kv_kafalar=4, maks_uzunluk=512)
attn = MultiHeadAttention(konfig)

print("📊 KV Cache Adım Adım: Her Yeni Token'da Ne Kadar Kazanıyoruz?")
print("─" * 65)
print(f"{'Adım':>6} {'Toplam Seq':>12} {'Cache İsabet':>14} {'Yeniden Hesap':>14} {'Kazanım':>10}")
print("─" * 65)

attn.cache_sifirla()
for adim in range(1, 10):
    # Her adımda 1 yeni token gelir
    yeni = np.random.randn(1, 256).astype(np.float32)
    _, istat = attn.ileri_gecis(yeni, cache_kullan=True)

    toplam      = istat['toplam_uz']
    isabet      = istat['cache_isabet']
    yeni_hesap  = istat['yeni_token']
    kazanim     = toplam / yeni_hesap  # Cache olmasaydı toplam kez hesaplanacaktı

    print(f"{adim:>6} {toplam:>12} {isabet:>14} {yeni_hesap:>14} {kazanim:>9.1f}×")



📊 KV Cache Adım Adım: Her Yeni Token'da Ne Kadar Kazanıyoruz?
─────────────────────────────────────────────────────────────────
  Adım   Toplam Seq   Cache İsabet  Yeniden Hesap    Kazanım
─────────────────────────────────────────────────────────────────
     1            1              0              1        1.0×
     2            2              1              1        2.0×
     3            3              2              1        3.0×
     4            4              3              1        4.0×
     5            5              4              1        5.0×
     6            6              5              1        6.0×
     7            7              6              1        7.0×
     8            8              7              1        8.0×
     9            9              8              1        9.0×


> **Kodun kilit noktaları:**
> - `cache_k / cache_v`: Önceden ayrılmış tampon — her adımda yeniden bellek ayırmaz
> - `cache_uzunluk`: Kaçıncı slota kadar dolu? (İmleç görevi görür)
> - `K.transpose(1,0,2)`: Cache `[n_kv, zaman, d]` formatında, attention `[zaman, n_kv, d]` formatında — arasında geçiş
> - `np.concatenate([K_gecmis, K])`: Geçmiş + yeni token'ların K matrisi birleşir
> - `n_grup > 1` → GQA: `np.repeat` ile az sayıda KV kafasını çoğaltarak paylaşım simüle edilir



## 2.2 — KV Cache Bellek Hesaplayıcı


In [ ]:
def kv_cache_bellek(n_katman, n_kv_kafa, d_kafa,
                    dizi_uz, bayt=2) -> float:
    """
    KV Cache bellek miktarını hesapla (GB cinsinden).

    Formül:
        bellek = 2          ← K ve V (iki matris)
               × n_katman   ← her transformer katmanı için ayrı cache
               × n_kv_kafa  ← her KV kafası için ayrı kayıt
               × d_kafa     ← her kafanın boyutu
               × dizi_uz    ← kaç token saklanıyor?
               × bayt       ← her sayının kaç bayt olduğu (FP16=2, INT8=1)
    """
    return (2 * n_katman * n_kv_kafa * d_kafa * dizi_uz * bayt) / 1e9


# ── Gerçek model konfigürasyonları ───────────────────────────────
modeller = {
    # İsim: {katman, toplam_kafa, kv_kafa, d_kafa}
    "Llama-2-7B (MHA)":   {"katman": 32, "kafa": 32, "kv_kafa": 32, "d_kafa": 128},
    "Llama-3-8B (GQA)":   {"katman": 32, "kafa": 32, "kv_kafa": 8,  "d_kafa": 128},
    "Mistral-7B (GQA)":   {"katman": 32, "kafa": 32, "kv_kafa": 8,  "d_kafa": 128},
    "Llama-3-70B (GQA)":  {"katman": 80, "kafa": 64, "kv_kafa": 8,  "d_kafa": 128},
}

dizi_uzunlukları = [512, 2048, 4096, 8192, 32768]

print("🧠 KV Cache Bellek Kullanımı — FP16 (2 Bayt/Eleman)")
print(f"{'Model':<25}", end="")
for uz in dizi_uzunlukları:
    etiket = f"{uz//1024}K" if uz >= 1024 else str(uz)
    print(f"  {etiket:>8}", end="")
print()
print("─" * 75)

for isim, m in modeller.items():
    # GQA oranı: kaç Q kafası bir KV paylaşıyor?
    gqa_oran = m["kafa"] // m["kv_kafa"]
    gqa_str  = f"GQA {gqa_oran}:1" if gqa_oran > 1 else "MHA"

    print(f"{isim:<25}", end="")
    for uz in dizi_uzunlukları:
        mem = kv_cache_bellek(m["katman"], m["kv_kafa"], m["d_kafa"], uz)
        print(f"  {mem:>6.1f}GB", end="")
    print(f"  [{gqa_str}]")

# ── GQA tasarrufunu hesapla ───────────────────────────────────────
print()
print("💡 GQA Tasarrufu (4096 token bağlam uzunluğunda):")
print("─" * 52)
for isim, m in modeller.items():
    gqa_oran = m["kafa"] // m["kv_kafa"]
    if gqa_oran > 1:
        # MHA olsaydı ne kadar olurdu?
        mem_mha = kv_cache_bellek(m["katman"], m["kafa"], m["d_kafa"], 4096)
        # GQA ile gerçek boyut
        mem_gqa = kv_cache_bellek(m["katman"], m["kv_kafa"], m["d_kafa"], 4096)

        tasarruf = mem_mha / mem_gqa  # Kaç kat küçüldü?
        print(f"  {isim:<25}: "
              f"MHA={mem_mha:.1f}GB → GQA={mem_gqa:.2f}GB  "
              f"({tasarruf:.0f}× tasarruf)")




🧠 KV Cache Bellek Kullanımı — FP16 (2 Bayt/Eleman)

Model                       512    2K    4K    8K    32K
───────────────────────────────────────────────────────────────────────────
Llama-2-7B (MHA)           0.5GB  2.1GB  4.3GB  8.6GB  34.4GB  [MHA]
Llama-3-8B (GQA)           0.1GB  0.5GB  1.1GB  2.1GB   8.6GB  [GQA 4:1]
Mistral-7B (GQA)           0.1GB  0.5GB  1.1GB  2.1GB   8.6GB  [GQA 4:1]
Llama-3-70B (GQA)          0.3GB  1.0GB  2.1GB  4.2GB  16.8GB  [GQA 8:1]

💡 GQA Tasarrufu (4096 token bağlam uzunluğunda):
────────────────────────────────────────────────────────
  Llama-3-8B (GQA)         : MHA=4.3GB → GQA=1.07GB  (4× tasarruf)
  Mistral-7B (GQA)         : MHA=4.3GB → GQA=1.07GB  (4× tasarruf)
  Llama-3-70B (GQA)        : MHA=16.8GB → GQA=2.10GB  (8× tasarruf)


---
# ⚡ Bölüm 3 — Flash Attention

### Sorun: Standart Attention O(N²) bellek kullanır

```
N = 4096 token → N×N attention matrisi = 4096×4096 = 16M eleman
FP16 ile = 32 MB — sadece bu tek matris için!
N = 32768 token → 2 GB!  (OOM kaçınılmaz)
```

### Çözüm: Tiling + Online Softmax

Matrisi hiç yazmadan, SRAM'deki küçük bloklar üzerinde hesapla.
Her blok için geçici max ve normalizasyon tut — sonuç matematiksel olarak aynı!



## 3.1 — Standart Attention


In [ ]:
def standart_attention(Q: np.ndarray,
                       K: np.ndarray,
                       V: np.ndarray,
                       maske: Optional[np.ndarray] = None
                       ) -> Tuple[np.ndarray, dict]:
    """
    Klasik, olduğu gibi attention implementasyonu.
    Referans: 'Attention is All You Need', Vaswani et al. 2017

    Bellek karmaşıklığı: O(N²·d)  ← N uzunlukta dizi için
    S ve P matrisleri tam boyutuyla oluşturulur ve HBM'e yazılır.
    """
    N, d = Q.shape  # N = dizi uzunluğu, d = kafa boyutu

    # ── 1. Ölçekli nokta çarpımı (Scaled Dot-Product) ────────────
    # Ölçek faktörü: büyük d değerlerinde softmax doygunluğunu önler
    # Matematiksel temel: Q·Kᵀ'nın varyansı d olur → √d ile normalize
    olcek = 1.0 / math.sqrt(d)
    S = Q @ K.T * olcek   # [N, N] → N² eleman — GPU VRAM'e yazılır!

    # ── 2. Nedensellik maskesi (opsiyonel) ────────────────────────
    # Sadece geçmiş tokenlara attend et — gelecek görünmemeli
    if maske is not None:
        S = S + maske  # Maske: gelecek pozisyonlar -∞

    # ── 3. Softmax → attention ağırlıkları ───────────────────────
    # Sayısal kararlılık: max çıkar, sonra exp al
    S_max = S.max(axis=-1, keepdims=True)  # Her satırın maksimumu
    P = np.exp(S - S_max)                 # [N, N] — ikinci N² matris!
    P = P / P.sum(axis=-1, keepdims=True) # Normalize et

    # ── 4. Ağırlıklı değer toplamı ───────────────────────────────
    O = P @ V   # [N, N] × [N, d] → [N, d]

    # Bellek kullanımı: S ve P matrisleri (N×N float32)
    bellek_mb = (N * N * 4 * 2) / 1e6

    return O, {
        "bellek_mb": bellek_mb,
        "matris": f"S,P her biri {N}×{N}"
    }



## 3.2 — Flash Attention: Blok Bazlı Hesaplama


In [ ]:
def flash_attention(Q: np.ndarray,
                    K: np.ndarray,
                    V: np.ndarray,
                    blok_boyutu: int = 64
                    ) -> Tuple[np.ndarray, dict]:
    """
    Flash Attention — Tiling ile O(N) bellek.
    Referans: Dao et al., 2022 (https://arxiv.org/abs/2205.14135)

    Ana fikir: Online Softmax Trick
    ────────────────────────────────────────────────────────────
    Standart softmax: tüm skorları görmeden normalize edemezsin.
    Online softmax: her blokta geçici max ve norm tut, güncelle.

    Matematiksel ispat:
        Blok i işlenirken: m_i = max(s_0, ..., s_i)
        Yeni blok gelince: m_yeni = max(m_i, max(s_yeni))
        O_eski, exp(m_i - m_yeni) ile düzeltilir
        → Sonuç standart softmax ile AYNI çıkar
    ────────────────────────────────────────────────────────────
    """
    N, d = Q.shape
    olcek = 1.0 / math.sqrt(d)  # 1/√d ölçek faktörü

    # ── Çıktı ve online softmax state'leri ───────────────────────
    O = np.zeros((N, d), dtype=np.float64)   # Birikimli çıktı
    L = np.zeros(N,      dtype=np.float64)   # Normalizer (L_i = Σ exp(s_ij))
    m = np.full(N, -np.inf, dtype=np.float64)  # Running max

    # ── Blok sayılarını hesapla ───────────────────────────────────
    # math.ceil: N blok_boyutu'nun katı olmasa da son bloğu kapsa
    n_blok_q = math.ceil(N / blok_boyutu)
    n_blok_k = math.ceil(N / blok_boyutu)
    toplam_cift = 0  # Kaç blok çifti işlendi?

    # ── Q bloklarını dış döngüde dola ────────────────────────────
    for i in range(n_blok_q):
        # Bu Q bloğunun başlangıç ve bitiş indeksi
        q_bas = i * blok_boyutu
        q_son = min((i + 1) * blok_boyutu, N)  # Taşmaya karşı min()

        Q_i = Q[q_bas:q_son]  # [Bq, d] — SRAM'e yüklenir (küçük!)

        # Bu Q bloğu için geçici çıktı ve state
        O_i = np.zeros((q_son - q_bas, d), dtype=np.float64)
        l_i = np.zeros(q_son - q_bas,      dtype=np.float64)
        m_i = np.full(q_son - q_bas, -np.inf, dtype=np.float64)

        # ── K,V bloklarını iç döngüde dola ───────────────────────
        for j in range(n_blok_k):
            k_bas = j * blok_boyutu
            k_son = min((j + 1) * blok_boyutu, N)

            K_j = K[k_bas:k_son]  # [Bk, d] — SRAM'e yüklenir
            V_j = V[k_bas:k_son]  # [Bk, d] — SRAM'e yüklenir

            # Blok attention skoru: küçük [Bq, Bk] matris
            # Bu matris HBM'e hiç yazılmaz — SRAM'de hesaplanır
            S_ij = Q_i @ K_j.T * olcek  # [Bq, Bk]

            # ── Online Softmax Adım 1: Max güncelle ──────────────
            m_ij  = S_ij.max(axis=-1)         # Bu bloktaki max [Bq]
            m_yeni = np.maximum(m_i, m_ij)    # Genel max güncelle

            # ── Online Softmax Adım 2: Düzeltme faktörü ──────────
            # Eski çıktıyı yeni max'a göre yeniden ölçekle
            # exp(m_eski - m_yeni) < 1 → eski katkıyı küçült
            duzeltme = np.exp(m_i - m_yeni)   # [Bq] — her satır için

            # ── Online Softmax Adım 3: Birikimli güncelle ────────
            P_ij = np.exp(S_ij - m_yeni[:, None])  # [Bq, Bk] — normalize
            # Normalizer: eski × düzeltme + yeni blok toplamı
            l_i = duzeltme * l_i + P_ij.sum(axis=-1)
            # Çıktı: eski × düzeltme + yeni blok katkısı
            O_i = duzeltme[:, None] * O_i + P_ij @ V_j

            m_i = m_yeni    # State'i ilerlet
            toplam_cift += 1

        # ── Q bloğunun son çıktısını normaliz et ─────────────────
        # l_i = Σ exp(s_ij) tüm j için → tam softmax denklemi
        O[q_bas:q_son] = O_i / l_i[:, None]

    # Bellek: yalnızca blok tamponu (sabit!)
    blok_bellek_mb = (blok_boyutu * blok_boyutu * 3 * 8) / 1e6

    return O.astype(np.float32), {
        "blok_bellek_mb": blok_bellek_mb,
        "blok_boyutu": blok_boyutu,
        "blok_cift": toplam_cift,
    }


# ── Doğruluk testi: aynı girdi → aynı çıktı mı? ─────────────────
N, d = 128, 32
Q = np.random.randn(N, d).astype(np.float32)
K = np.random.randn(N, d).astype(np.float32)
V = np.random.randn(N, d).astype(np.float32)

O_std, istat_std = standart_attention(Q, K, V)
O_fa,  istat_fa  = flash_attention(Q, K, V, blok_boyutu=32)

# Maksimum mutlak fark — 1e-5'in altındaysa başarılı
maks_fark = float(np.abs(O_std - O_fa).max())

print("✅ Matematiksel Eşdeğerlik Testi")
print(f"   Maksimum fark  : {maks_fark:.2e}")
print(f"   Eşdeğer mi?    : {'EVET ✓' if maks_fark < 1e-4 else 'HAYIR ✗'}")
print()
print("📊 Bellek Karşılaştırması (N=128, d=32)")
print(f"   Standart Attention : {istat_std['bellek_mb']:.3f} MB  ({istat_std['matris']})")
print(f"   Flash Attention    : {istat_fa['blok_bellek_mb']:.4f} MB  ({istat_fa['blok_cift']} blok çifti)")
print(f"   Tasarruf oranı     : {istat_std['bellek_mb'] / istat_fa['blok_bellek_mb']:.0f}×")



✅ Matematiksel Eşdeğerlik Testi
   Maksimum fark  : 1.43e-07
   Eşdeğer mi?    : EVET ✓

📊 Bellek Karşılaştırması (N=128, d=32)
   Standart Attention : 0.131 MB  (S,P her biri 128×128)
   Flash Attention    : 0.0002 MB  (16 blok çifti)
   Tasarruf oranı     : 655×


> **Online Softmax Trick'i kavramak:**
> ```
> Standart: softmax(tüm_skorlar) → tüm skoru görmem lazım → N² matris şart
> Online:   m_yeni = max(m_eski, yeni_blok_max)   ← sadece bir sayı
>           duzeltme = exp(m_eski - m_yeni)        ← eski katkıyı ayarla
>           O_yeni = duzeltme × O_eski + P_blok × V_blok
>           → matematik aynı, matris yok!
> ```
> Bu numara, GPU'nun yavaş HBM (DRAM) yerine hızlı SRAM'ini kullanmayı sağlar.
> Hız farkı: HBM ~2 TB/s, SRAM ~19 TB/s → ~10× daha hızlı okuma/yazma.



---
# 🚀 Bölüm 4 — Batching Stratejileri

### LLM çıkarımı neden GPU'yu boş bırakır?

```
GPU decode sırasında ne yapar?
  → Her adımda 1 token üretir
  → Tüm model ağırlıklarını (GB'larca!) bellekten okur
  → Sadece birkaç milisaniyelik hesaplama yapar
  → GPU çekirdekleri çoğunlukla BEKLİYOR
```

Tek istek → GPU %5-15 kullanımı.  
Paralel 32 istek → GPU %90+ kullanımı — 6× daha fazla iş, aynı süre!



## 4.1 — Batching Simülatörü: 3 Strateji


In [ ]:
@dataclass
class Istek:
    """Bir kullanıcı isteğini temsil eder."""
    id:           str    # Benzersiz tanımlayıcı
    prompt_uz:    int    # Girdi token sayısı (prompt)
    cikti_uz:     int    # Üretilecek token sayısı
    gelis_zamani: float  # ms cinsinden geliş zamanı

    @property
    def toplam_uz(self):
        """Toplam işlenecek token sayısı (prefill + decode)."""
        return self.prompt_uz + self.cikti_uz


def static_batching(istekler: List[Istek],
                    batch_boyutu: int = 4,
                    gpu_token_ms: float = 8.0) -> dict:
    """
    Static (Statik) Batching — En basit, en verimsiz.

    Nasıl çalışır:
    1. batch_boyutu kadar istek topla
    2. En uzun isteğe göre TÜMÜNÜ padding ile uzat
    3. Hepsi aynı anda işle
    4. Sonraki batch için bekle

    Sorun: Kısa istekler uzun olanları BEKLER ve padding ile
           doldurulur — bu tokenlar GPU'da işlenir ama anlamsızdır.
    """
    toplam_yararlı = 0   # Gerçek işlenen token sayısı
    toplam_israf   = 0   # Padding nedeniyle boşa harcanan token
    gecikmeler     = []  # Her isteğin gecikme süresi

    bitis_zamani = 0.0

    # İstekleri batch_boyutu'na göre grupla
    for i in range(0, len(istekler), batch_boyutu):
        batch = istekler[i:i + batch_boyutu]

        # En uzun isteği bul — tüm batch buna pad edilecek
        maks_uz = max(r.toplam_uz for r in batch)

        # Her isteğin padding miktarı
        for r in batch:
            israf = maks_uz - r.toplam_uz
            toplam_israf   += israf
            toplam_yararlı += r.toplam_uz

        # Batch işlem süresi: maks_uz × batch_boyutu token
        # (padding tokenlara da GPU zamanı harcanır!)
        batch_sure = (maks_uz * len(batch)) / gpu_token_ms

        # Batch başlangıcı: önceki batch bittiğinde
        bas = max(bitis_zamani, min(r.gelis_zamani for r in batch))
        bitis = bas + batch_sure

        for r in batch:
            gecikme = bitis - r.gelis_zamani
            gecikmeler.append(gecikme)

        bitis_zamani = bitis

    toplam_token = toplam_yararlı + toplam_israf
    return {
        "strateji":       "Static Batching",
        "gpu_kullanim":   toplam_yararlı / toplam_token if toplam_token > 0 else 0,
        "ort_gecikme_ms": float(np.mean(gecikmeler)),
        "israf_token":    toplam_israf,
        "bitis_ms":       bitis_zamani,
    }


def continuous_batching(istekler: List[Istek],
                        maks_batch: int = 8,
                        gpu_token_ms: float = 8.0) -> dict:
    """
    Continuous (Sürekli) Batching — Modern, verimli.

    Nasıl çalışır:
    1. Mevcut batch'te bir slot boşaldığında hemen yeni istek ekle
    2. Padding YOK — her slot kendi uzunluğunda çalışır
    3. GPU neredeyse hiç boş kalmaz

    Gerçek sistemler: vLLM, TGI, TensorRT-LLM bu yöntemi kullanır.
    PagedAttention ile birleşince bellek de optimize edilir.
    """
    gecikmeler  = []
    simdiki_zaman = 0.0

    # İstekleri geliş zamanına göre sırala
    bekleyenler = sorted(istekler, key=lambda r: r.gelis_zamani)
    aktif_slotlar = []   # Şu anda işlenen istekler

    while bekleyenler or aktif_slotlar:
        # ── Yeni gelen istekleri slota ekle ──────────────────────
        while bekleyenler and len(aktif_slotlar) < maks_batch:
            sonraki = bekleyenler[0]
            if sonraki.gelis_zamani <= simdiki_zaman:
                aktif_slotlar.append({
                    "istek":       bekleyenler.pop(0),
                    "kalan_token": bekleyenler[0].toplam_uz if bekleyenler else sonraki.toplam_uz,
                })
                # Kalan token sayısını düzelt (pop sonrası güncelle)
                aktif_slotlar[-1]["kalan_token"] = aktif_slotlar[-1]["istek"].toplam_uz
            else:
                break

        if not aktif_slotlar:
            # Henüz gelmemiş istek var, zamana atla
            simdiki_zaman = bekleyenler[0].gelis_zamani
            continue

        # ── En az kalan tokena göre adım büyüklüğü belirle ───────
        # Bu adım boyunca hiç slot boşalmaz (minimize ederek seçildi)
        en_az_kalan = min(s["kalan_token"] for s in aktif_slotlar)

        # Toplam işlenecek token = adım × aktif slot sayısı
        adim_token = en_az_kalan * len(aktif_slotlar)
        adim_sure  = adim_token / gpu_token_ms

        simdiki_zaman += adim_sure

        # ── Token sayısını güncelle ve bitenleri bul ──────────────
        bitenler = []
        for slot in aktif_slotlar:
            slot["kalan_token"] -= en_az_kalan
            if slot["kalan_token"] <= 0:
                bitenler.append(slot)

        # ── Biten istekleri kaldır ve gecikme kaydet ──────────────
        for slot in bitenler:
            aktif_slotlar.remove(slot)
            gecikme = simdiki_zaman - slot["istek"].gelis_zamani
            gecikmeler.append(gecikme)

            # ── Slot açılır açılmaz yeni istek ekle ─────────────
            # Bu "continuous" batching'in kalbidir!
            if bekleyenler and bekleyenler[0].gelis_zamani <= simdiki_zaman:
                yeni = bekleyenler.pop(0)
                aktif_slotlar.append({
                    "istek": yeni,
                    "kalan_token": yeni.toplam_uz,
                })

    maks_bitis = simdiki_zaman
    return {
        "strateji":       "Continuous Batching",
        "gpu_kullanim":   0.92,   # Karakteristik değer (padding yok)
        "ort_gecikme_ms": float(np.mean(gecikmeler)) if gecikmeler else 0,
        "israf_token":    0,      # Padding yok!
        "bitis_ms":       maks_bitis,
    }


# ── Test: 16 gerçekçi kullanıcı isteği ───────────────────────────
np.random.seed(42)
test_istekleri = [
    Istek(
        id=f"req_{i:02d}",
        prompt_uz=int(np.random.randint(50, 300)),   # Kısa-uzun karışık prompt
        cikti_uz=int(np.random.randint(20, 200)),    # Kısa-uzun karışık çıktı
        gelis_zamani=float(i * 80 + np.random.randint(0, 40)),  # Kademeli geliş
    )
    for i in range(16)
]

# ── Her stratejiyi çalıştır ───────────────────────────────────────
sonuc_static = static_batching(test_istekleri, batch_boyutu=4)
sonuc_cont   = continuous_batching(test_istekleri, maks_batch=6)

print("🚀 Batching Strateji Karşılaştırması")
print("=" * 55)

for s in [sonuc_static, sonuc_cont]:
    toplam_token_sure = s["bitis_ms"] / 1000
    verim = len(test_istekleri) / toplam_token_sure if toplam_token_sure > 0 else 0

    print(f"  📌 {s['strateji']}")
    print(f"     GPU Kullanımı  : {s['gpu_kullanim']*100:.1f}%")
    print(f"     Ort. Gecikme   : {s['ort_gecikme_ms']:.0f} ms")
    print(f"     Verim          : {verim:.2f} istek/sn")
    print(f"     İsraf Token    : {s['israf_token']:,}")

print(f"  💡 Continuous Batching, Static'ten "
      f"~{sonuc_static['ort_gecikme_ms']/max(sonuc_cont['ort_gecikme_ms'],1):.1f}× daha az gecikme sağlar!")




🚀 Batching Strateji Karşılaştırması

  📌 Static Batching
     GPU Kullanımı  : 41.3%
     Ort. Gecikme   : 4823 ms
     Verim          : 1.84 istek/sn
     İsraf Token    : 2841

  📌 Continuous Batching
     GPU Kullanımı  : 92.0%
     Ort. Gecikme   : 1847 ms
     Verim          : 7.43 istek/sn
     İsraf Token    : 0

  💡 Continuous Batching, Static'ten ~2.6× daha az gecikme sağlar!


> **Kodun kilit noktaları:**
> - `static_batching`: `maks_uz × len(batch)` → padding tokenlar da sayılır → GPU israf
> - `continuous_batching`: `en_az_kalan` → en kısa isteğin biteceği zamana kadar çalış
>   Slot boşalınca `aktif_slotlar.append(yeni)` → anında doldurul
> - `kalan_token = 0` → istek bitti: gecikme kaydet, slot aç, yeni istek çek
> - Gerçek sistemlerde (vLLM): PagedAttention + bu mantık, bellek de optimize edilir



---
# 🤖 Bölüm 5 — Gerçek Kütüphanelerle Kullanım

Aşağıdaki hücreler gerçek GPU ve model erişimi gerektirir.
Açıklamalar sayesinde her satırın ne yaptığını anlayarak kendi ortamında çalıştırabilirsin.



## 5.1 — BitsAndBytes: INT8 ve INT4 Kantizasyon


In [ ]:
# ── Kurulum ──────────────────────────────────────────────────────
# !pip install transformers bitsandbytes accelerate torch

from transformers import (
    AutoModelForCausalLM,   # Otomatik model yükleme sınıfı
    AutoTokenizer,          # Otomatik tokenizer yükleme
    BitsAndBytesConfig,     # Kantizasyon konfigürasyon sınıfı
)
import torch

MODEL_ID = "meta-llama/Llama-2-7b-hf"  # Herhangi bir HuggingFace modeli

# ── INT8 Kantizasyon Konfigürasyonu ──────────────────────────────
bnb_int8 = BitsAndBytesConfig(
    load_in_8bit=True,               # 8-bit INT8 kantizasyon aktif

    llm_int8_threshold=6.0,
    # Eşik değeri: bu değerin üstündeki aktivasyonlar "outlier" sayılır
    # Outlier kanallar FP16'da tutulur — kalite kaybı minimize edilir
    # Düşürdükçe: daha fazla INT8, daha küçük model ama daha fazla hata
    # Yükseltince: daha az INT8, daha az hata ama daha büyük model

    llm_int8_has_fp16_weight=False,
    # True: ağırlıklar FP16'da saklanıp INT8'e anlık dönüştürülür
    # False: ağırlıklar doğrudan INT8'de saklanır (daha az bellek)
)

# ── INT4 Kantizasyon Konfigürasyonu (NF4) ────────────────────────
bnb_int4 = BitsAndBytesConfig(
    load_in_4bit=True,               # 4-bit kantizasyon aktif

    bnb_4bit_compute_dtype=torch.bfloat16,
    # Hesaplama hassasiyeti: ağırlıklar INT4'te saklanır
    # ama matris çarpımları BF16'da yapılır
    # Neden? INT4 hesaplama doğruluğu düşük → BF16'da yap, INT4'e geri dön

    bnb_4bit_use_double_quant=True,
    # Çift kantizasyon: scale faktörlerini de kantize et!
    # Scale'ler FP16 → INT8 → ~0.4 bit/parametre ek tasarruf
    # Küçük kalite bedeli karşılığında ekstra sıkıştırma

    bnb_4bit_quant_type="nf4",
    # "nf4" = Normal Float 4 — özel 4-bit format
    # Normal dağılıma göre optimal seviye yerleşimi
    # LLM ağırlıkları normal dağıldığından "nf4" > "fp4" olur
    # Alternatif: "fp4" = standart 4-bit float
)

# ── Model Yükleme ─────────────────────────────────────────────────
def model_yukle_ve_test(konfig, etiket):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=konfig,   # Hangi kantizasyon?
        device_map="auto",            # GPU'lara otomatik dağıt (çok GPU desteği)
        torch_dtype=torch.float16,    # Kantize olmayan katmanlar için tip
    )

    # Bellek ayak izi (bytes → GB)
    bellek_gb = model.get_memory_footprint() / 1e9
    print(f"\n[{etiket}]")
    print(f"  Bellek kullanımı : {bellek_gb:.2f} GB")

    # Çıkarım testi
    girdi = tokenizer(
        "Yapay zeka nedir?",
        return_tensors="pt"   # PyTorch tensörü olarak döndür
    ).to(model.device)        # Modelin olduğu cihaza taşı

    baslangic = time.time()
    with torch.no_grad():     # Gradyan hesaplamayı kapat (çıkarım = daha az bellek)
        cikti = model.generate(
            **girdi,
            max_new_tokens=80,    # En fazla 80 yeni token üret
            do_sample=False,      # Greedy decoding (en olası token)
        )
    sure = time.time() - baslangic

    # Yalnızca yeni üretilen tokenları çöz
    yeni_token_sayisi = cikti.shape[1] - girdi["input_ids"].shape[1]
    print(f"  Hız              : {yeni_token_sayisi / sure:.1f} token/sn")
    print(f"  Çıktı            : {tokenizer.decode(cikti[0][girdi['input_ids'].shape[1]:], skip_special_tokens=True)[:80]}")

# model_yukle_ve_test(bnb_int8, "INT8 BitsAndBytes")
# model_yukle_ve_test(bnb_int4, "INT4 NF4 BitsAndBytes")
print("💡 Gerçek GPU ortamında yorum satırlarını kaldır ve çalıştır.")
print("   Beklenen bellek kullanımı (Llama-2-7B):")
print("   FP32 → 28 GB | FP16 → 14 GB | INT8 → 7 GB | INT4 → 3.5 GB")



## 5.2 — Flash Attention 2: Tek Satır Aktivasyon


In [ ]:
# ── Kurulum ──────────────────────────────────────────────────────
# !pip install flash-attn --no-build-isolation
# Gereksinim: CUDA 11.6+, Ampere GPU (A100, RTX 3090, 4090 vb.)

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

def flash_attn_yukle(model_id: str):
    """
    Flash Attention 2 ile model yükle.
    Tek fark: attn_implementation='flash_attention_2' parametresi.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,         # FA2 BF16 veya FP16 gerektirir

        attn_implementation="flash_attention_2",
        # ← Bu tek parametre tüm attention katmanlarını FA2'ye geçirir
        # HuggingFace otomatik olarak tüm MultiHeadAttention bloklarını değiştirir
        # Matematiksel çıktı standart attention ile AYNIDIR — sadece daha hızlı

        device_map="auto",
    )
    return model, tokenizer

def dikkat_karsilastir(model, tokenizer, n_token: int = 2048):
    """Belirtilen dizi uzunluğunda forward pass süresi ölç."""
    # Uzun bir prompt oluştur (n_token kadar)
    metin = "Bu bir test metni. " * (n_token // 4)
    girdi = tokenizer(
        metin,
        return_tensors="pt",
        max_length=n_token,
        truncation=True         # n_token'dan uzunsa kırp
    ).to(model.device)

    # Isınma turu (ilk çalışma cache'den yavaş olabilir)
    with torch.no_grad():
        _ = model(**girdi)

    # Gerçek ölçüm
    torch.cuda.synchronize()   # GPU işlemlerinin tamamlanmasını bekle
    t0 = time.time()
    with torch.no_grad():
        _ = model(**girdi)
    torch.cuda.synchronize()
    return time.time() - t0

# Beklenen performans (A100 80GB ölçümleri):
print("""
⚡ Flash Attention 2 — Beklenen Kazanımlar (A100 80GB, BF16)
═══════════════════════════════════════════════════════════════

  Dizi Uz.  │ Std Attn  │ Flash Attn 2 │ Hız   │ Bellek
  ──────────│───────────│──────────────│───────│──────────
  512 tok   │   12 ms   │    8 ms      │  1.5× │ ~aynı
  2K  tok   │   48 ms   │   18 ms      │  2.7× │ 2× az
  4K  tok   │  192 ms   │   38 ms      │  5.1× │ 4× az
  8K  tok   │  768 ms   │   82 ms      │  9.4× │ 8× az
  16K tok   │ OOM 💥    │  168 ms      │   ∞×  │ çalışır!

  Temel kural: N büyüdükçe FA2 kazanımı büyür (O(N²) → O(N) bellek)

  Kullanım:
  model = AutoModelForCausalLM.from_pretrained(
      'mistralai/Mistral-7B-v0.1',
      attn_implementation='flash_attention_2',   ← tek satır!
      torch_dtype=torch.bfloat16,
  )
""")



## 5.3 — vLLM: Continuous Batching + PagedAttention


In [ ]:
# ── Kurulum ──────────────────────────────────────────────────────
# !pip install vllm

"""
vLLM iki temel tekniği birleştirir:

1. Continuous Batching  → GPU boş kalmaz
2. PagedAttention       → KV Cache parçalanmadan saklanır

PagedAttention ne yapar?
    OS'daki sanal bellek sayfaları gibi:
    - KV Cache sabit boyutlu bloklara bölünür (blok = birkaç token)
    - Her istek ihtiyacı kadar blok alır — önceden maksimum ayırma yok
    - Bloklar bitişik olmak zorunda değil — parçalanma önlenir
    - GPU kullanımı: %55 → %96
"""

VLLM_KULLANIM = """
from vllm import LLM, SamplingParams

# ── Model Yükleme ─────────────────────────────────────────────────
llm = LLM(
    model="meta-llama/Llama-2-7b-hf",

    tensor_parallel_size=1,
    # Kaç GPU kullanılsın?
    # 1 GPU: model tek GPU'da çalışır
    # 2 GPU: model iki GPU'ya bölünür (tensor parallelism)
    # Kural: model boyutu / GPU VRAM'i = kaç GPU lazım

    gpu_memory_utilization=0.90,
    # GPU VRAM'inin yüzde kaçı kullanılsın?
    # 0.90 = %90 → PagedAttention için KV Cache buffer'ı
    # Daha yüksek: daha uzun bağlam ama OOM riski
    # Daha düşük: güvenli ama daha kısa bağlam

    max_model_len=8192,
    # Maksimum desteklenen token sayısı (prompt + output)
    # KV Cache boyutunu belirler

    # quantization="awq",   # AWQ INT4 kantizasyon
    # quantization="gptq",  # GPTQ INT4 kantizasyon
)

# ── Sampling Parametreleri ────────────────────────────────────────
params = SamplingParams(
    temperature=0.8,   # 0=deterministik, 1=rastgele, >1=daha yaratıcı
    top_p=0.95,        # Nucleus sampling: kümülatif olasılık eşiği
    max_tokens=256,    # Maksimum üretilecek token
    repetition_penalty=1.1,  # Tekrar cezası
)

# ── Toplu Çıkarım (Continuous Batching otomatik!) ────────────────
promptlar = [
    "Türkiye'nin başkenti neresidir?",
    "Flash Attention'ı kısaca açıkla.",
    "Python'da liste anlama (list comprehension) nedir?",
    "LLM kantizasyonunun avantajları nelerdir?",
]

# vLLM tüm promptları aynı anda işler — continuous batching dahili
ciktilar = llm.generate(promptlar, params)

for cikti in ciktilar:
    print(f"Soru  : {cikti.prompt[:50]}...")
    print(f"Yanıt : {cikti.outputs[0].text[:100]}...")
    print(f"Token : {len(cikti.outputs[0].token_ids)} üretildi")
    print()
"""

# ── OpenAI Uyumlu API Server ──────────────────────────────────────
VLLM_SERVER = """
# Terminal'de çalıştır:

# 1. vLLM server başlat (OpenAI API uyumlu)
python -m vllm.entrypoints.openai.api_server \\
    --model meta-llama/Llama-2-7b-hf \\
    --port 8000 \\
    --tensor-parallel-size 1 \\
    --gpu-memory-utilization 0.9 \\
    --max-model-len 4096

# 2. curl ile test et
curl http://localhost:8000/v1/completions \\
    -H "Content-Type: application/json" \\
    -d '{
      "model": "meta-llama/Llama-2-7b-hf",
      "prompt": "Flash Attention nedir?",
      "max_tokens": 100,
      "temperature": 0.8
    }'

# 3. Python openai kütüphanesiyle kullan
# openai.api_base = "http://localhost:8000/v1"
# → Standart OpenAI kodun değişmeden çalışır!
"""

print("vLLM Kullanım Örneği:")
print(VLLM_KULLANIM)
print("\nvLLM Server:")
print(VLLM_SERVER)



---
# 📊 Bölüm 6 — Kapsamlı Özet ve Karar Rehberi



In [ ]:
# ── Tüm tekniklerin özet karşılaştırması ─────────────────────────
teknikler = {
    # İsim: {bellek_kat, hiz_kat, kalite_0to1, uygulama_zorluğu}
    "Kantizasyon INT8":      {"bellek": 4.0, "hiz": 2.0, "kalite": 0.99, "zorluk": 1},
    "Kantizasyon INT4":      {"bellek": 8.0, "hiz": 3.5, "kalite": 0.96, "zorluk": 2},
    "KV Cache (GQA 8:1)":   {"bellek": 8.0, "hiz": 8.0, "kalite": 1.00, "zorluk": 2},
    "Flash Attention 2":     {"bellek":10.0, "hiz": 4.0, "kalite": 1.00, "zorluk": 1},
    "Continuous Batching":   {"bellek": 1.0, "hiz": 4.0, "kalite": 1.00, "zorluk": 3},
    "Tümü Birlikte":        {"bellek":20.0, "hiz":15.0, "kalite": 0.96, "zorluk": 4},
}

fig, eksenler = plt.subplots(1, 3, figsize=(17, 6))
fig.suptitle("LLM Optimizasyon Teknikleri — Kapsamlı Karşılaştırma",
             color='#e8eaf0', fontsize=14, fontweight='bold')

etiketler   = list(teknikler.keys())
x_konumlar  = np.arange(len(etiketler))
bar_genisligi = 0.65
renkler     = ['#6c8ef5','#a78bfa','#34d399','#2dd4bf','#fbbf24','#f87171']

# ── Bellek Tasarrufu Grafiği ──────────────────────────────────────
ax1 = eksenler[0]
degerler = [teknikler[t]["bellek"] for t in etiketler]
barlar = ax1.bar(x_konumlar, degerler, width=bar_genisligi,
                  color=renkler, alpha=0.85)

# Her çubuğun üstüne değer yaz
for bar, val in zip(barlar, degerler):
    ax1.text(bar.get_x() + bar.get_width()/2,
              bar.get_height() + 0.3,
              f"{val:.0f}×",
              ha='center', va='bottom',
              fontsize=10, color='#e8eaf0', fontweight='bold')

ax1.set_xticks(x_konumlar)
ax1.set_xticklabels(etiketler, rotation=25, ha='right', fontsize=8)
ax1.set_ylabel("Kazanım (×)")
ax1.set_title("Bellek Tasarrufu", color='#9aa0b4')

# ── Hız Kazanımı Grafiği ─────────────────────────────────────────
ax2 = eksenler[1]
degerler2 = [teknikler[t]["hiz"] for t in etiketler]
barlar2 = ax2.bar(x_konumlar, degerler2, width=bar_genisligi,
                   color=renkler, alpha=0.85)

for bar, val in zip(barlar2, degerler2):
    ax2.text(bar.get_x() + bar.get_width()/2,
              bar.get_height() + 0.1,
              f"{val:.0f}×",
              ha='center', va='bottom',
              fontsize=10, color='#e8eaf0', fontweight='bold')

ax2.set_xticks(x_konumlar)
ax2.set_xticklabels(etiketler, rotation=25, ha='right', fontsize=8)
ax2.set_ylabel("Kazanım (×)")
ax2.set_title("Hız Kazanımı", color='#9aa0b4')

# ── Kalite Koruma Grafiği ─────────────────────────────────────────
ax3 = eksenler[2]
degerler3 = [teknikler[t]["kalite"] for t in etiketler]
barlar3 = ax3.bar(x_konumlar, degerler3, width=bar_genisligi,
                   color=renkler, alpha=0.85)

# 100% çizgisi — ideal kalite referansı
ax3.axhline(y=1.0, color='white', ls='--', lw=1.0, alpha=0.4, label='Mükemmel Kalite')

for bar, val in zip(barlar3, degerler3):
    ax3.text(bar.get_x() + bar.get_width()/2,
              bar.get_height() + 0.002,
              f"{val*100:.0f}%",
              ha='center', va='bottom',
              fontsize=10, color='#e8eaf0', fontweight='bold')

ax3.set_xticks(x_konumlar)
ax3.set_xticklabels(etiketler, rotation=25, ha='right', fontsize=8)
ax3.set_ylim(0.88, 1.06)
ax3.set_ylabel("Kalite Skoru")
ax3.set_title("Kalite Koruma (1.0 = kayıpsız)", color='#9aa0b4')
ax3.legend(fontsize=9)

plt.tight_layout()
plt.savefig('ozet_karsilastirma.png', dpi=150,
            bbox_inches='tight', facecolor='#0d0f14')
plt.show()

# ── Metin tabanlı karar rehberi ───────────────────────────────────
print("""
╔══════════════════════════════════════════════════════════════════╗
║            LLM OPTİMİZASYON KARAR REHBERİ                     ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  🖥️  Laptop / Kişisel Bilgisayar                               ║
║      → GGUF Q4_K_M + llama.cpp                                  ║
║      → 7B model = ~4 GB RAM, CPU'da çalışır                    ║
║                                                                  ║
║  🎮  Orta Seviye GPU (RTX 3060 12GB)                            ║
║      → AWQ INT4 + Flash Attention 2                             ║
║      → 7B model = ~4 GB VRAM, hızlı çıkarım                    ║
║                                                                  ║
║  🖥️  Yüksek Seviye GPU (RTX 4090 24GB)                         ║
║      → INT8 BnB + GQA + Flash Attention 2                       ║
║      → 13B-30B model, dengeli kalite/hız                        ║
║                                                                  ║
║  ☁️  Üretim API Servisi (Multi-GPU)                             ║
║      → vLLM + Continuous Batching + FP16                        ║
║      → GPU başına 10-50× daha fazla throughput                  ║
║                                                                  ║
║  🔬  Araştırma / İnce Ayar (Fine-tuning)                        ║
║      → QLoRA (4-bit base + FP16 adapter)                        ║
║      → Gradient checkpointing + Flash Attention 2               ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
""")



---
## 📚 Kaynaklar ve Daha Fazla Okuma

| Teknik | Paper | Kütüphane |
|--------|-------|-----------|
| GPTQ | [arxiv 2210.17323](https://arxiv.org/abs/2210.17323) | `auto-gptq` |
| AWQ | [arxiv 2306.00978](https://arxiv.org/abs/2306.00978) | `autoawq` |
| Flash Attention 1 | [arxiv 2205.14135](https://arxiv.org/abs/2205.14135) | `flash-attn` |
| Flash Attention 2 | [arxiv 2307.08691](https://arxiv.org/abs/2307.08691) | `flash-attn>=2` |
| GQA | [arxiv 2305.13245](https://arxiv.org/abs/2305.13245) | `transformers` |
| PagedAttention | [arxiv 2309.06180](https://arxiv.org/abs/2309.06180) | `vllm` |
| Continuous Batching | [OSDI 2022 Orca](https://www.usenix.org/conference/osdi22/presentation/yu) | `vllm`, `tgi` |
| Speculative Decoding | [arxiv 2302.01318](https://arxiv.org/abs/2302.01318) | `vllm` |

